In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from utils.metrics import qini_score

from models.gbdt_uplift_model import TwoStageGradientBoostingUpliftClassifier

DATASET_PATH = os.path.join(os.getcwd(), "data")

def run_experiment(dataset_alias, fold_id=0):
    print(f"\n{'='*50}")
    print(f"RUNNING EXPERIMENT: {dataset_alias} (Fold {fold_id})")
    print(f"{'='*50}")
    
    # 1. Load Dữ liệu
    folder = os.path.join(DATASET_PATH, f'{dataset_alias}_{fold_id}')
    train_path = os.path.join(folder, 'train.pkl')
    test_path = os.path.join(folder, 'test.pkl')
    
    if not os.path.exists(train_path):
        print(f"[Error] Data not found at {train_path}.")
        return

    train = joblib.load(train_path)
    test = joblib.load(test_path)
    
    print(train)
    
    X_train, y_train, t_train = train['X'], train['y'], train['t']
    X_test, y_test, t_test = test['X'], test['y'], test['t']

    # Xác định có bao nhiêu loại Treatment (loại bỏ số 0 là Control)
    available_treatments = np.unique(t_train)
    treatment_ids = available_treatments[available_treatments != 0]
    
    print(f"Train Shape: {X_train.shape}")
    print(f"Treatments Found: {treatment_ids}")

    results = {}
    
    # 2. Vòng lặp train riêng cho từng cặp (Control vs Treatment_i)
    for t_id in treatment_ids:
        print(f"\n--- Processing Treatment {t_id} vs Control ---")
        
        # 2a. Lọc dữ liệu Train (Chỉ lấy dòng Control + Treatment hiện tại)
        mask_train = np.isin(t_train, [0, t_id])
        X_tr_sub = X_train[mask_train]
        y_tr_sub = y_train[mask_train]
        t_tr_sub = t_train[mask_train]
        
        # Map lại nhãn treatment về 0 và 1 (Control=0, Treat_i=1)
        t_tr_binary = (t_tr_sub == t_id).astype(np.int32)
        
        # 2b. Khởi tạo & Train Model
        print("  > Training Model...")
        model = TwoStageGradientBoostingUpliftClassifier(
            learning_rate=0.05,
            max_depth=6,
            n_estimators=300,        # Số lượng cây (bạn có thể tăng lên 1000 nếu có GPU mạnh)
            uplift_ensemble_weight=0.5, # Trọng số kết hợp (50% tin vào Outcome, 50% tin vào Uplift trực tiếp)
            verbose=100              # In log mỗi 100 vòng
        )
        
        # Fit model
        model.fit(X_tr_sub, y_tr_sub, t_tr_binary)
        
        # 2c. Dự đoán & Đánh giá trên tập Test
        # Cũng lọc tập test tương ứng
        mask_test = np.isin(t_test, [0, t_id])
        X_test_sub = X_test[mask_test]
        y_test_sub = y_test[mask_test]
        t_test_sub = t_test[mask_test]
        t_test_binary = (t_test_sub == t_id).astype(np.int32)
        
        # Predict trả về Uplift Score
        pred_uplift = model.predict(X_test_sub)
        
        # Tính Qini Score
        score = qini_score(y_test_sub, pred_uplift, t_test_binary)
        results[f'Treatment_{t_id}'] = score
        print(f"  > Done. Qini Score: {score:.4f}")

    return results


In [5]:
# --- Chạy thử Dataset Hillstrom (Fold 0) ---
# Hillstrom là dataset Marketing Email thực tế (Men's Email vs Women's Email)
scores_hillstrom = run_experiment('hillstrom', fold_id=0)
print("\nFinal Results (Hillstrom):", scores_hillstrom)

# --- Chạy thử Dataset Synthetic (Fold 0) ---
# Dataset giả lập với 6 loại treatment khác nhau
# scores_synth = run_experiment('synth1', fold_id=0)
# print("\nFinal Results (Synthetic):", scores_synth)


RUNNING EXPERIMENT: hillstrom (Fold 0)
{'X': array([[ 10.  ,   2.  , 142.44, ...,   0.  ,   0.  ,   0.  ],
       [  6.  ,   3.  , 329.08, ...,   1.  ,   1.  ,   1.  ],
       [  7.  ,   2.  , 180.65, ...,   0.  ,   1.  ,   1.  ],
       ...,
       [  6.  ,   1.  ,  29.99, ...,   2.  ,   1.  ,   0.  ],
       [  1.  ,   5.  , 552.94, ...,   0.  ,   1.  ,   2.  ],
       [  1.  ,   4.  , 472.82, ...,   0.  ,   0.  ,   1.  ]],
      dtype=float32), 'y': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 't': array([2, 0, 2, ..., 1, 2, 1]), 'trg': array([[nan, nan,  0.],
       [ 0., nan, nan],
       [nan, nan,  0.],
       ...,
       [nan,  0., nan],
       [nan, nan,  0.],
       [nan,  0., nan]], dtype=float32), 'p': array([[0.49979275, 0.5027319 ],
       [0.49979275, 0.5027319 ],
       [0.49979275, 0.5027319 ],
       ...,
       [0.49979275, 0.5027319 ],
       [0.49979275, 0.5027319 ],
       [0.49979275, 0.5027319 ]], dtype=float32)}
Train Shape: (51200, 8)
Treatments Found

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from models.gbdt_uplift_model import TwoStageGradientBoostingUpliftClassifier

DATASET_PATH = os.path.join(os.getcwd(), "data")

# --- 1. COPY LOGIC TÍNH TOÁN CỦA TÁC GIẢ (Từ visualization.py) ---

def get_mse_author_logic(y_true, t_true, pred_uplift):
    """
    Tính MSE theo cách của tác giả:
    So sánh dự đoán (pred_uplift) với 'effect' thật (chỉ có trong Synthetic Data).
    Công thức: (y_true - pred_uplift)^2 trung bình trên từng treatment (aka mean())
    y_true: được coi là các giá trị thực (numeric values) đại diện cho cường độ tác động
        1: chắc chắn mua
        0: không quan tâm (mua hoặc không mua bất kể có coupon hay không)
        -1: sleeping dogs: Ghét bị làm phiền (nhận coupon -> chắc chắn ko mua)
    Returns: Array of MSE cho từng treatment
    """
    # Trong Synthetic data, t_true có nhiều treatment (1, 2, 3...)
    # Hàm này tính MSE cho từng treatment
    
    n_treatments = t_true.max()
    mse_scores = []
    
    for t in range(1, n_treatments + 1):
        # Lấy các dòng thuộc treatment t
        mask = (t_true == t)
        
        if np.sum(mask) == 0:
            mse_scores.append(0.0)
            continue
            
        y_effect_true = y_true[mask] # Đây phải là cột 'effect' (True Uplift)
        
        # Nếu pred_uplift là 1 cột (binary), ta dùng trực tiếp
        # Nếu pred_uplift là nhiều cột (multi-treatment), ta lấy cột t-1
        if pred_uplift.ndim > 1 and pred_uplift.shape[1] >= t:
             p_pred = pred_uplift[mask, t-1]
        else:
             p_pred = pred_uplift[mask] # Binary case
             
        # Tính MSE
        mse_val = ((y_effect_true - p_pred) ** 2).mean()
        mse_scores.append(mse_val)
        
    return np.array(mse_scores)

def qini_score_simple(y, pred, t):
    """
    Tính Qini/AUUC đơn giản cho 1 fold.
    (Đây là giá trị tác giả gọi là 'AUUC' trong bảng)
    """
    # Sắp xếp giảm dần theo pred
    order = np.argsort(pred)[::-1]
    y_sorted = y[order]
    t_sorted = t[order]
    
    # Tính Qini Area
    y_t = np.cumsum(y_sorted * (t_sorted == 1))
    y_c = np.cumsum(y_sorted * (t_sorted == 0))
    n_t = np.cumsum(t_sorted == 1)
    n_c = np.cumsum(t_sorted == 0)
    
    # Tránh chia cho 0
    nt_total = n_t[-1] if n_t[-1] > 0 else 1
    nc_total = n_c[-1] if n_c[-1] > 0 else 1
    
    qini_curve = y_t - y_c * (nt_total / nc_total)
    
    # Area under curve
    # Đơn giản hóa bằng tổng (giống một số thư viện metric) hoặc trapz
    # Tác giả dùng công thức qini cụ thể trong thư viện sklift/causalml, 
    # ở đây ta dùng xấp xỉ diện tích.
    return np.sum(qini_curve) / (len(y) ** 2) # Chuẩn hóa đơn giản

# --- 2. HÀM CHẠY THỰC NGHIỆM 5 FOLDS ---

def run_5fold_experiment(dataset_alias, model_params):
    print(f"Running Experiment on {dataset_alias} (5 Folds)...")
    
    auuc_list = [] # Lưu AUUC của 5 folds
    mse_list = []  # Lưu MSE của 5 folds (nếu có)
    
    # Giả lập 5 folds bằng cách load các file dữ liệu tác giả đã tạo sẵn
    # Folder: data/synth1_0, data/synth1_1, ...
    
    for i in range(5):
        print(f"  > Processing Fold {i}...")
        
        # Load Data
        folder = os.path.join(DATASET_PATH, f'{dataset_alias}_{i}')
        if not os.path.exists(folder):
            print(f"    [!] Missing data for fold {i}. Run data_preparation.py first.")
            continue
            
        train = joblib.load(os.path.join(folder, 'train.pkl'))
        test = joblib.load(os.path.join(folder, 'test.pkl'))
        
        # Xử lý dữ liệu Synthetic (có cột 'effect' thật để tính MSE)
        has_effect = 'effect' in test
        
        # Lấy Treatment 1 làm ví dụ (Binary Case: Treat 1 vs Control)
        # Trong code tác giả, họ loop qua tất cả treatment.
        # Ở đây ta demo với Treatment 1.
        target_treat_id = 1
        
        # Lọc dữ liệu Train (Control + Treat 1)
        mask_tr = np.isin(train['t'], [0, target_treat_id])
        X_tr = train['X'][mask_tr]
        y_tr = train['y'][mask_tr]
        t_tr = (train['t'][mask_tr] == target_treat_id).astype(int)
        
        # Train Model
        model = TwoStageGradientBoostingUpliftClassifier(**model_params)
        model.fit(X_tr, y_tr, t_tr)
        
        # Lọc dữ liệu Test
        mask_te = np.isin(test['t'], [0, target_treat_id])
        X_te = test['X'][mask_te]
        y_te = test['y'][mask_te]
        t_te = (test['t'][mask_te] == target_treat_id).astype(int)
        
        # Predict
        uplift_pred = model.predict(X_te)
        
        # --- TÍNH METRIC ---
        
        # 1. AUUC (Qini)
        score = qini_score_simple(y_te, uplift_pred, t_te)
        auuc_list.append(score)
        
        # 2. MSE (Chỉ tính được nếu biết Uplift thật - Synthetic Data)
        if has_effect:
            # Lấy cột effect thật của test set
            effect_true = test['effect'][mask_te]
            # Tính MSE: Mean((True - Pred)^2)
            mse_val = ((effect_true - uplift_pred)**2).mean()
            mse_list.append(mse_val)
            
    return np.array(auuc_list), np.array(mse_list)

# --- 3. TẠO BẢNG KẾT QUẢ (FORMAT GIỐNG TÁC GIẢ) ---

def format_result(values):
    if len(values) == 0:
        return "N/A"
    # Format: Mean ± Std
    mean = np.mean(values)
    std = np.std(values, ddof=1) # ddof=1 cho sample std
    return f"{mean:.4f} ± {std:.4f}"

if __name__ == '__main__':
    # Cấu hình Model giống "GBDT Average" (weight=0.5)
    params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 300,
        'uplift_ensemble_weight': 0.5, # Đây là weight tạo nên sự khác biệt
        'verbose': False
    }
    
    # 1. Chạy trên Synth1 (Có MSE)
    auucs, mses = run_5fold_experiment('synth1', params)
    
    # 2. Tạo DataFrame kết quả
    results = {
        'Model': ['My GBDT Implementation'],
        'AUUC (Treat 1)': [format_result(auucs)],
        'MSE (Treat 1)': [format_result(mses)]
    }
    
    df = pd.DataFrame(results)
    
    print("\n=== REPLICATION RESULTS ===")
    print(df.to_string(index=False))

Running Experiment on synth1 (5 Folds)...
  > Processing Fold 0...
  > Processing Fold 1...
  > Processing Fold 2...
  > Processing Fold 3...
  > Processing Fold 4...

=== REPLICATION RESULTS ===
                 Model  AUUC (Treat 1)   MSE (Treat 1)
My GBDT Implementation 0.0132 ± 0.0024 0.0285 ± 0.0014

(So sánh với bảng trong ResultsMain.ipynb dòng 'GBDT Average' cột '1')
